# Scraper for Regulations and Guidance

Link: https://www.mas.gov.sg/regulation/regulations-and-guidance?topics=Anti-Money%20Laundering&page=1&rows=All

## Obtain All the Links

In [1]:
"""
MAS Regulatory Pages Scraper Utilities
"""

import re
from typing import List, Dict
from bs4 import BeautifulSoup
from selenium import webdriver
from urllib.parse import urljoin
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


def mas_regulations_scraper(url: str) -> List[Dict]:
    """
    Scrapes regulations and guidance from MAS search results page.

    Args:
        url: The URL of the MAS regulations and guidance search page

    Returns:
        A list of regulation items, each containing:
        - title: Title of the regulation
        - url: Link to the regulation page
        - category: Category/tag of the regulation
        - date: Publication/update date
        - summary: Brief summary
        - topics: List of related topics
        - consultation_fields: Optional consultation information

    Example:
        >>> items = mas_regulations_scraper("https://www.mas.gov.sg/regulation/regulations-and-guidance?topics=Anti-Money%20Laundering&page=1&rows=All")
        >>> print(len(items))
        136
    """
    # Setup Chrome driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.get(url)
    html_content = driver.page_source
    driver.quit()

    # Prettify the HTML content
    soup = BeautifulSoup(html_content, "html.parser")

    base_url = "https://www.mas.gov.sg"

    items = []
    for li in soup.find_all("li", class_="mas-search-page__result"):
        # title + link
        a = li.select_one(".ola-field-title a.mas-link") or li.find(
            "a", class_="mas-link"
        )
        title = a.get_text(" ", strip=True) if a else None
        href = urljoin(base_url, a["href"]) if a and a.has_attr("href") else None

        # category / tag
        tag_el = li.select_one(".mas-tag__text")
        category = tag_el.get_text(strip=True) if tag_el else None

        # date (try to find a DD Month YYYY pattern inside ancillaries)
        date = None
        anc = li.select_one(".mas-ancillaries")
        if anc:
            text = anc.get_text(" ", strip=True)
            m = re.search(r"\d{1,2}\s+[A-Za-z]+\s+\d{4}", text)
            date = m.group(0) if m else text.strip()

        # summary / body
        body_p = li.select_one(".mas-search-card__body p")
        summary = body_p.get_text(" ", strip=True) if body_p else None

        # topics / footer tags (may be multiple)
        topics = []
        for foot_a in li.select("footer a.mas-link .mas-link__text"):
            t = foot_a.get_text(" ", strip=True)
            if t and t not in topics:
                topics.append(t)

        # consultation fields (optional)
        consultation = {}
        for cf in li.select(".consultation-field"):
            label = cf.contents[0].strip() if cf.contents else ""
            span = cf.select_one("span")
            if span:
                consultation[label.rstrip(":")] = span.get_text(" ", strip=True)

        items.append(
            {
                "title": title,
                "url": href,
                "category": category,
                "date": date,
                "summary": summary,
                "topics": topics,
                "consultation_fields": consultation or None,
            }
        )

    return items

## Extract History

In [3]:
"""
Notice History Scraper for MAS Regulatory Pages
"""

from typing import List, Dict
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


def notice_history_scraper(url: str) -> List[Dict]:
    """
    Scrapes the amendment history from a MAS notice page.

    Args:
        url: The URL of the MAS notice page to scrape

    Returns:
        A list of amendment entries, each containing:
        - date: The date of the amendment
        - documents: List of documents with title and url

    Example:
        >>> entries = notice_history_scraper("https://www.mas.gov.sg/regulation/notices/notice-314")
        >>> print(entries)
        [
            {
                "date": "01 Jan 2024",
                "documents": [
                    {
                        "title": "Amendment Notice",
                        "url": "/path/to/document.pdf"
                    }
                ]
            }
        ]
    """
    # Setup Chrome driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.get(url)
    html_content = driver.page_source
    driver.quit()

    soup = BeautifulSoup(html_content, "html.parser")

    base_url = "https://www.mas.gov.sg"

    # Find the description list containing amendment notes
    dl = soup.find("dl", class_="mas-description-list")

    if not dl:
        return []

    # Find all div elements that contain dt/dd pairs
    amendment_entries = []

    for div in dl.find_all("div", recursive=False):
        dt = div.find("dt")
        dd = div.find("dd")

        if dt and dd:
            # Extract date
            date = dt.get_text(strip=True)

            # Extract all PDF links from dd
            documents = []
            for link in dd.find_all("a", class_="mas-link"):
                # Extract title
                title_span = link.find("span", class_="mas-link__text")
                title = title_span.get_text(strip=True) if title_span else ""

                # Extract URL
                doc_url = link.get("href", "")

                if doc_url:
                    doc_url = urljoin(base_url, doc_url)

                document = {"title": title, "url": doc_url}

                documents.append(document)

            # Only add entry if it has documents
            if documents:
                amendment_entries.append({"date": date, "documents": documents})

    return amendment_entries

In [4]:
mas_regulations = mas_regulations_scraper(
    "https://www.mas.gov.sg/regulation/regulations-and-guidance?rows=10&sort=mas_date_tdt%20desc&page=1&topics=Anti-Money%20Laundering"
)

In [5]:
notice_history = notice_history_scraper(
    "https://www.mas.gov.sg/regulation/notices/notice-314"
)

In [6]:
notice_history

[{'date': '30 Jun 2025',
  'documents': [{'title': 'MAS Notice 314 (Cancellation) Notice 2025',
    'url': 'https://www.mas.gov.sg/-/media/mas-media-library/regulation/notices/id/notice-314/mas-notice-314-cancellation-notice-2025.pdf'},
   {'title': 'MAS Notice 314 dated 24 Apr 2015 (last revised 1 March 2022)',
    'url': 'https://www.mas.gov.sg/-/media/mas-media-library/regulation/notices/id/notice-314/notice-314-last-revised-on-1-march-2022.pdf'}]},
 {'date': '01 Mar 2022',
  'documents': [{'title': 'Notice 314 (Amendment) 2022',
    'url': 'https://www.mas.gov.sg/-/media/mas-media-library/regulation/notices/id/notice-314/notice-314-amendment-2022.pdf'}]},
 {'date': '28 Jun 2021',
  'documents': [{'title': 'Notice 314 (Amendment) 2021',
    'url': 'https://www.mas.gov.sg/-/media/mas-media-library/regulation/notices/id/notice-314/mas-314_tracked_28-jun-2021.pdf'}]},
 {'date': '30 Nov 2015',
  'documents': [{'title': 'Notice 314 (Amendment) 2015',
    'url': 'https://www.mas.gov.sg/-/

# Full Running

In [5]:
for regulation in mas_regulations_scraper(
    "https://www.mas.gov.sg/regulation/regulations-and-guidance?rows=10&sort=mas_date_tdt%20desc&page=1&topics=Anti-Money%20Laundering"
):  # this is in json format
    if regulation["category"] == "Notices":
        notice_history_scraper(regulation["url"])
    elif regulation["category"] == "Guidelines":
        notice_history_scraper(regulation["url"])
    else:
        pass

## PDF difference

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup

def get_prettified_html(url: str) -> str:
    # Configure headless Chrome
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    # Initialize WebDriver
    driver = webdriver.Chrome(options=chrome_options)
    driver.get(url)
    
    # Get page source
    html = driver.page_source
    
    # Close the browser
    driver.quit()
    
    # Prettify the HTML
    soup = BeautifulSoup(html, "html.parser")
    return soup.prettify()


In [ ]:
import requests
import os
import time
from datetime import datetime
from pathlib import Path
from selenium.webdriver.chrome.options import Options


def get_file_size(url):
    """Get the size of a file via HEAD request"""
    try:
        response = requests.head(url, timeout=10, allow_redirects=True)
        if "content-length" in response.headers:
            return int(response.headers["content-length"])
        else:
            # Fallback: try GET with stream to check size
            response = requests.get(url, stream=True, timeout=10)
            return int(response.headers.get("content-length", 0))
    except Exception as e:
        print(f"Error getting size for {url}: {e}")
        return 0


def setup_driver(download_dir):
    """Setup Chrome driver with download preferences"""
    chrome_options = Options()

    # Set download directory
    prefs = {
        "download.default_directory": str(download_dir),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "plugins.always_open_pdf_externally": True,  # Automatically download PDFs
        "safebrowsing.enabled": True,
    }
    chrome_options.add_experimental_option("prefs", prefs)

    # Optional: Run in headless mode (uncomment if you don't want to see the browser)
    # chrome_options.add_argument("--headless")

    driver = webdriver.Chrome(options=chrome_options)
    return driver


def wait_for_download(download_dir, timeout=30):
    """Wait for download to complete"""
    seconds = 0
    dl_wait = True
    while dl_wait and seconds < timeout:
        time.sleep(1)
        dl_wait = False
        for fname in os.listdir(download_dir):
            if fname.endswith(".crdownload") or fname.endswith(".tmp"):
                dl_wait = True
        seconds += 1
    return seconds < timeout


def download_pdf_selenium(driver, url, output_path):
    """Download a PDF using Selenium"""
    try:
        print(f"Downloading: {output_path.name}")
        driver.get(url)

        # Wait for download to complete
        download_dir = output_path.parent
        if wait_for_download(download_dir):
            # Find the downloaded file and rename it
            files = sorted(
                Path(download_dir).glob("*"), key=os.path.getmtime, reverse=True
            )
            for file in files:
                if file.suffix == ".pdf" and file.name != output_path.name:
                    # Rename the downloaded file
                    file.rename(output_path)
                    file_size = output_path.stat().st_size
                    print(f"✓ Downloaded {output_path.name} ({file_size:,} bytes)")
                    return True

            # File already exists with correct name
            if output_path.exists():
                file_size = output_path.stat().st_size
                print(f"✓ Downloaded {output_path.name} ({file_size:,} bytes)")
                return True

        print(f"✗ Download timeout for {output_path.name}")
        return False
    except Exception as e:
        print(f"✗ Error downloading {output_path.name}: {e}")
        return False


def format_date_for_filename(date_str):
    """Convert date string to filename-friendly format (YYYY-MM-DD)"""
    try:
        date_obj = datetime.strptime(date_str, "%d %b %Y")
        return date_obj.strftime("%Y-%m-%d")
    except Exception as e:
        print(f"Error parsing date '{date_str}': {e}")
        return date_str.replace(" ", "_").replace("/", "-")


def main(data=notice_history):
    # Create output directory
    output_dir = Path("mas_pdfs").resolve()
    output_dir.mkdir(exist_ok=True)

    print(f"Starting PDF downloads to {output_dir}/\n")

    # Setup Selenium driver
    driver = setup_driver(output_dir)

    try:
        for entry in data:
            date_str = entry["date"]
            documents = entry["documents"]

            print(f"\nProcessing date: {date_str}")
            print(f"Found {len(documents)} document(s)")

            # Format the date for filename
            filename_date = format_date_for_filename(date_str)

            if len(documents) == 1:
                # Single document - download it
                doc = documents[0]
                filename = output_dir / f"{filename_date}.pdf"
                print(f"Single document: {doc['title']}")
                download_pdf_selenium(driver, doc["url"], filename)

            elif len(documents) == 2:
                # Two documents - download the smaller one
                doc1, doc2 = documents

                print(f"Document 1: {doc1['title']}")
                size1 = get_file_size(doc1["url"])
                print(f"  Size: {size1:,} bytes")

                print(f"Document 2: {doc2['title']}")
                size2 = get_file_size(doc2["url"])
                print(f"  Size: {size2:,} bytes")

                # Download the smaller one
                filename = output_dir / f"{filename_date}.pdf"
                if size1 > 0 and size2 > 0:
                    if size1 < size2:
                        print(f"Downloading smaller document (Document 1)")
                        download_pdf_selenium(driver, doc1["url"], filename)
                    else:
                        print(f"Downloading smaller document (Document 2)")
                        download_pdf_selenium(driver, doc2["url"], filename)
                elif size1 > 0:
                    print(f"Downloading Document 1 (couldn't get size for Document 2)")
                    download_pdf_selenium(driver, doc1["url"], filename)
                else:
                    print(f"Downloading Document 2 (couldn't get size for Document 1)")
                    download_pdf_selenium(driver, doc2["url"], filename)

            else:
                print(f"Unexpected number of documents: {len(documents)}")

            # Small delay between downloads
            time.sleep(2)

    finally:
        # Close the browser
        driver.quit()

    print(f"\n✓ All downloads completed! Files saved to {output_dir}/")

In [22]:
main(notice_history)

Starting PDF downloads to /Users/javianng/TheCode/baerly_awake/backend_1/test/mas_pdfs/


Processing date: 30 Jun 2025
Found 2 document(s)
Document 1: MAS Notice 314 (Cancellation) Notice 2025
  Size: 20 bytes
Document 2: MAS Notice 314 dated 24 Apr 2015 (last revised 1 March 2022)
  Size: 20 bytes
Downloading: 2025-06-30.pdf
✓ Downloaded 2025-06-30.pdf (273,239 bytes)

Processing date: 01 Mar 2022
Found 1 document(s)
Single document: Notice 314 (Amendment) 2022
Downloading: 2022-03-01.pdf
✓ Downloaded 2022-03-01.pdf (282,179 bytes)

Processing date: 28 Jun 2021
Found 1 document(s)
Single document: Notice 314 (Amendment) 2021
Downloading: 2021-06-28.pdf
✓ Downloaded 2021-06-28.pdf (452,508 bytes)

Processing date: 30 Nov 2015
Found 1 document(s)
Single document: Notice 314 (Amendment) 2015
Downloading: 2015-11-30.pdf
✓ Downloaded 2015-11-30.pdf (34,489 bytes)

Processing date: 24 Apr 2015
Found 1 document(s)
Single document: Notice 314 dated 24 Apr 2015
Downloading: 2015-04-24.pdf
✓ Do

In [ ]:
from PyPDF2 import PdfReader
import difflib


def extract_text_from_pdf(pdf_path):
    """Extract all text from a PDF and return as a list of pages."""
    reader = PdfReader(pdf_path)
    pages_text = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages_text.append(text)
        else:
            pages_text.append("")
    return pages_text


def compare_pdfs(pdf1_path, pdf2_path):
    pdf1_text = extract_text_from_pdf(pdf1_path)
    pdf2_text = extract_text_from_pdf(pdf2_path)

    num_pages = max(len(pdf1_text), len(pdf2_text))
    print(
        f"Comparing {len(pdf1_text)} pages (PDF1) vs {len(pdf2_text)} pages (PDF2)...\n"
    )

    for i in range(num_pages):
        text1 = pdf1_text[i] if i < len(pdf1_text) else ""
        text2 = pdf2_text[i] if i < len(pdf2_text) else ""

        diff = difflib.unified_diff(
            text1.splitlines(),
            text2.splitlines(),
            fromfile=f"PDF1_page_{i+1}",
            tofile=f"PDF2_page_{i+1}",
            lineterm="",
        )

        diff_list = list(diff)
        if diff_list:
            print(f"\n--- Differences found on page {i+1} ---")
            for line in diff_list:
                print(line)
        else:
            print(f"Page {i+1}: No differences found.")


# Example usage
pdf1 = "mas_pdfs/2025-06-30.pdf"
pdf2 = "mas_pdfs/2022-03-01.pdf"
compare_pdfs(pdf1, pdf2)

: 